## Notebook Overview

This notebook will run the full VCD/MFCD Comparison. It initializes the environment, synchronizes the repository, prepares datasets, and runs both Task A (VCD reproduction) and Task B (MFCD improvement) on the MSCOCO POPE splits using LLaVA‑1.5. All results are saved to a unified output directory and summarized in a comparison table.

### What this notebook does

- Sets up the runtime, installs dependencies, and prepares COCO‑Val2014
- Runs Task A (Regular vs. VCD on random/popular POPE)
- Runs Task B (MFCD on random/popular POPE)
- Generates a final comparison table (Regular vs. VCD vs. MFCD)

### Outputs

- JSON result files for each split/method
- A formatted table with Accuracy, Precision, Recall, and F1 for all methods

In [1]:
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA Version:', torch.version.cuda)

Torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA Version: 12.8


In [ ]:
import os
from pathlib import Path

# Define paths
REPO_URL = 'https://github.com/maxheef/ECE209_Project1.git'
ROOT = "/content/VCD_project"
MY_TASKS = f"{ROOT}/myTasks"

# Clone if it doesn't exist
if not os.path.exists(ROOT):
    print("Cloning repository for the first time...")
    !git clone {REPO_URL} {ROOT}
else:
    print("Repository already exists.")

# Move to the scripts for configuration
os.chdir(MY_TASKS)
print(f"Current Directory: {os.getcwd()}")

# Verify files are visible
!ls


Repository already exists.
Current Directory: /content/VCD_project/myTasks
analysis.py  init_setup.py  __pycache__  sync_repo.py
hidden	     Main.ipynb     scripts	 tasks


## Setup 

In [35]:
# Infrastructure & Data (~8 mins first time, <1 sec after) ---
sys.path.append('/content/VCD_project/myTasks/scripts')
import sys

# Test to find which GPU configurations are available
import importlib.util
print("G4 Setup Found:", importlib.util.find_spec("setup_g4") is not None)
print("H100 Setup Found:", importlib.util.find_spec("setup_h100") is not None)

%run init_setup.py

G4 Setup Found: True
H100 Setup Found: True
--- Syncing Repository (main) ---
Repository synced successfully.
Conda environments already exist. Skipping hardware setup.
MSCOCO already exists. Skipping download.

[SUCCESS] All components are initialized and quiet.


## Sync Git

In [60]:
# Only need to run this if you do not want to run init_setup.py every time
%run sync_repo.py 

--- Syncing Repository (main) ---
Sync complete. Code is up to date.


In [37]:
# Load bins for Task A and B
from pathlib import Path
VCD_PY = Path('/tmp/vcd_bin.txt').read_text()
MFCD_PY = Path('/tmp/mfcd_bin.txt').read_text()

## Task A - Reproduction (Midterm) 
#### Reproduce the evaluation setting for: 
- **Dataset source**: MSCOCO (POPE split) 
- **Model**: LLaVA-1.5 
- **Methods**: Regular decoding (baseline) vs. VCD decoding 
- **POPE** settings: random and popular 

In [38]:
# Run Task A
# Compare Regular with VCD on Random/Popular splits
print("Running Task A (VCD)...")
!{VCD_PY} tasks/task_a_vcd.py

Running Task A (VCD)...

>>> Task A: Running random | regular
/usr/local/miniconda/envs/vcd310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100% 2/2 [00:03<00:00,  1.87s/it]
100% 3000/3000 [04:01<00:00, 12.45it/s]

>>> Task A: Running random | vcd
/usr/local/miniconda/envs/vcd310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100% 2/2 [00:03<00:00,  1.86s/it]
100% 3000/3000 [07:40<00:00,  6.51it/s]

>>> Task A: Running popular | regular
/usr/local/miniconda/envs/vcd310/lib/python3.10/site-

## Task B - Improvement beyond VCD w/ MFCD (Final)
##### Identify and attempt at least one method that extends or improves VCD and aims to further increase performance (e.g., accuracy) on the same evaluation setting(s) you used in Task A.
##### MFCD (Multi-Frequency Contrastive Decoding, EMNLP 2025) was utilized for improvement

In [14]:
# Run Task B
# Compute MFCD on Random/Popular splits
print("Running Task B (MFCD)...") 
!{MFCD_PY} tasks/task_b_mfcd.py

Running Task B (MFCD)...
processor_config.json: 100% 173/173 [00:00<00:00, 2.82MB/s]
chat_template.json: 100% 701/701 [00:00<00:00, 14.7MB/s]
chat_template.jinja: 100% 674/674 [00:00<00:00, 14.0MB/s]
preprocessor_config.json: 100% 505/505 [00:00<00:00, 10.4MB/s]
tokenizer_config.json: 1.45kB [00:00, 16.9MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 1.22MB/s]
tokenizer.json: 3.62MB [00:00, 73.7MB/s]
added_tokens.json: 100% 41.0/41.0 [00:00<00:00, 851kB/s]
special_tokens_map.json: 100% 552/552 [00:00<00:00, 11.6MB/s]
Applying chat template: 100% 188/188 [00:05<00:00, 35.98it/s]
Processing batch data:  83% 156/188 [00:18<00:03, 10.53it/s]/usr/local/miniconda/envs/mfcd310/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Processing batch data: 100% 188/188 [00:23<00:00,  8.00it/s]
config.json: 100% 9

## Show Comparison Table

In [62]:
# Generate A Comparison Table for Regular, VCD, and MFCD
# Code for comparison table is in myTasks/analysis.py
import importlib
import analysis

OUT_DIR = '/content/VCD_project/output'
analysis.show_comparison_table(OUT_DIR)


/content/VCD_project/myTasks/analysis.py:73: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[reg_idx[0], 'Gain vs Regular (%)'] = '-'


### VQA Mitigation Comparison: Regular vs VCD vs MFCD

Split,Method,Accuracy,Precision,Recall,F1,Gain vs Regular (%)
random,Regular,0.8287,0.9199,0.7200,0.8078,-
random,VCD,0.8557,0.9369,0.7627,0.8409,+4.10%
random,MFCD,0.8677,0.8506,0.8920,0.8708,+7.80%
popular,Regular,0.8107,0.8795,0.7200,0.7918,-
popular,VCD,0.8357,0.8931,0.7627,0.8227,+3.91%
popular,MFCD,0.8220,0.7835,0.8900,0.8333,+5.25%
